# Repertoire Comparison Benchmark against CompAIRR

This notebook benchmarks repertoire comparison against the the state-of-the-art tool named CompAIRR ([Rognes et al.](https://doi.org/10.1093/bioinformatics/btac505)). That is, given a number of immune repertoires and a Hamming distance threshold, the algorithm computes the overlap between all pairs of repertoires. The overlap of two repertoires is defined as the count of sequence pairs (below a chosen threshold) found across these repertoires. The dataset is obtained from [Emerson et al](https://doi.org/10.1038/ng.3822).

The notebook is divided into 6 steps as follow:
1. __Configuration:__ select the number of experiment repeats and maximum dataset size. Note that the largest option of dataset size requires high-RAM VM which requires paid Google Colab account.
2. __Benchmark Setup:__ install dependencies and implementations of various algorithms and compile them if need be.
3. __Repertoire Comparison Benchmark:__ perform benchmark comparing Compairr on CPU, SymDel algorithm on CPU and XTNeighbor on GPU at threshold `d=1,2`.
4. __Result Download:__ download the benchmark measurement as csv file.

Warning: some sections take up to 1 hour to run. Indicative timings are provided for each step.

## 0. Configuration

In [ ]:
# @title Configure the runtime and number of experiment repeats. High RAM is only available in Colab's premium plan.
n_repeat = 1 # @param ["1", "10", "30"] {type:"raw"}
high_ram = False # @param {type:"boolean"}

## 1. Benchmark Setup (run time ~ 3 min)

install dependency

In [ ]:
! pip install -q pyrepseq

In [ ]:
import os.path
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import time

from airrutils import *

try:
    from google.colab import files
    colab = True
except ImportError:
    colab = False

clone the projects

In [ ]:
if not os.path.exists("compairr"):
    !git clone https://github.com/uio-bmi/compairr.git

if colab and not os.path.exists("XT-neighbor"):
    !git clone https://github.com/heartnetkung/XT-neighbor.git
    repo_path = "XT-neighbor/"
else:
    repo_path = "../"

compile XTNeighbor-streaming

In [ ]:
! mkdir -p {repo_path}xtneighbor_streaming/build
! cd {repo_path}xtneighbor_streaming/build; cmake ..;make

compile Compairr

In [ ]:
!cd compairr; make

prepare repertoire info

In [ ]:
def read_info():
  ans = pd.read_csv(f'{repo_path}data/info.csv')
  end = np.cumsum(ans['count'])
  ans['start'] = np.concatenate(([0],end[:-1]))
  ans['end'] = end
  return ans

info = read_info()
info

prepare input data

In [ ]:
! mkdir -p tmp

In [ ]:
N_FILES=5

def read_input():
  for i in range(1,N_FILES+1):
    ! unzip -n {repo_path}/data/emerson_rep"$i".zip -d tmp
  reps = []
  for i in range(1,N_FILES+1):
    reps.append(pd.read_csv(f'tmp/emerson_rep{i}.txt'))
  return pd.concat(reps,ignore_index=True)

data = read_input()
print(data.head())

## 2. Repertoire Comparison Benchmark (run time ~ 30 min at n_repeat=1,high_ram=False and ~ 120 min at n_repeat=1,high_ram=True)

check GPU availability

In [ ]:
import subprocess
try:
  subprocess.run(["nvidia-smi"], capture_output=True, text=True)
except Exception as e:
  raise Exception("GPU required")

input preparation code

In [ ]:
## Test single CPU run time for CompAIRR and SymScan
n_cpu = os.cpu_count()
os.environ['RAYON_NUM_THREADS'] = str(n_cpu)
print(f"Using {n_cpu} CPU cores for SymScan and CompAIRR")

benchmarking code

In [ ]:
sizes = [1,2,4,8,16,32,64]
algorithms = {
    'symdel':symdel_overlap,
    'symscan':symscan_overlap,
    'xt_streaming': xt_neighbor_overlap,
    'compairr': lambda *args, **kwargs: compairr_overlap(*args, **kwargs, n_cpu=n_cpu)
}
limits = {
    'symdel_1':16,
    'symdel_2':16,
    'symscan_1':64,
    'symscan_2':32,
    'xt_streaming_1':64,
    'xt_streaming_2':32,
    'compairr_1':64,
    'compairr_2':16,
}
if not high_ram:
    sizes = [1,2,4,8]

result_data = {'runtime':[],'algorithm':[],'n_sequence':[],'distance':[],'measure':[],'n_repertoire':[]}

def run_exp(distance, is_hamming):
    for i in range(n_repeat):
        for size in sizes:
            seq_info, reps = sample_repertoire(data,info,size,random_state=i)
            seqs, dup_counts, rep_sizes = prepare(seq_info,reps)
            _len = len(seqs)
            for alg_name in algorithms:
                limit = limits.get(f"{alg_name}_{distance}")
                if limit is not None and limit <size:
                    continue
                # compairr does not support distance=2 with levenshtein distance
                if (alg_name == 'compairr') and (distance == 2) and not is_hamming:
                    continue
                # perform
                start = time.time()
                algorithms[alg_name](distance,is_hamming,seqs, dup_counts, rep_sizes)
                end = time.time()

                # record
                print(f'{size:,}',_len,alg_name,i,round((end-start)*100)/100)
                result_data['runtime'].append(end-start)
                result_data['algorithm'].append(alg_name)
                result_data['n_sequence'].append(len(seqs))
                result_data['distance'].append(distance)
                result_data['measure'].append('hamming' if is_hamming else 'leven')
                result_data['n_repertoire'].append(size)

In [ ]:
run_exp(distance=1, is_hamming=True)

In [ ]:
run_exp(distance=1, is_hamming=False)   

In [ ]:
run_exp(distance=2, is_hamming=True)

In [ ]:
run_exp(distance=2, is_hamming=False)

## 3. Result Download (run time < 1 min)

In [ ]:
pd.DataFrame(result_data).to_csv('compairr_benchmark.csv')
if colab:
    files.download('compairr_benchmark.csv')